In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\sadat\.conda\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sadat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sadat\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
478,Just saw the World Preem of Fido at the Toront...,positive
736,"I went to see Random Hearts with 3 friends, an...",negative
535,"When I fist watched the movie, I said to mysel...",positive
50,Here's a real weirdo for you. It starts out wi...,positive
812,I first heard of this film when Patton Oswalt ...,negative


In [6]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
478,saw world preem fido toronto international fil...,positive
736,went see random heart friend first thought may...,negative
535,fist watched movie said myself so film made li...,positive
50,here s real weirdo you start another take off ...,positive
812,first heard film patton oswalt talked werewolf...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
negative    269
positive    231
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [14]:
df.head()

,review,sentiment
478,saw world preem fido toronto international fil...,positive
736,went see random heart friend first thought may...,negative
535,fist watched movie said myself so film made li...,positive
50,here s real weirdo you start another take off ...,positive
812,first heard film patton oswalt talked werewolf...,negative


In [15]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [16]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [18]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Sadat-Shakeeb/production-mlops-sentiment-analysis.mlflow')
dagshub.init(repo_owner='Sadat-Shakeeb', repo_name='mlops-sentiment-proj', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-03-13 16:40:53,549 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/Sadat-Shakeeb/mlops-sentiment-proj "HTTP/1.1 200 OK"


Initialized MLflow to track repo "Sadat-Shakeeb/mlops-sentiment-proj"

2026-03-13 16:40:53,556 - INFO - Initialized MLflow to track repo "Sadat-Shakeeb/mlops-sentiment-proj"


Repository Sadat-Shakeeb/mlops-sentiment-proj initialized!

2026-03-13 16:40:53,559 - INFO - Repository Sadat-Shakeeb/mlops-sentiment-proj initialized!


<Experiment: artifact_location='mlflow-artifacts:/5089f73b2d5e4683874cf2290480b625', creation_time=1773399657426, experiment_id='0', last_update_time=1773399657426, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [19]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, pos_label='positive')
        recall = recall_score(y_test, y_pred, pos_label='positive')
        f1 = f1_score(y_test, y_pred, pos_label='positive')

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-03-13 16:41:14,737 - INFO - Starting MLflow run...
2026-03-13 16:41:15,412 - INFO - Logging preprocessing parameters...
2026-03-13 16:41:16,546 - INFO - Initializing Logistic Regression model...
2026-03-13 16:41:16,548 - INFO - Fitting the model...
2026-03-13 16:41:16,583 - INFO - Model training complete.
2026-03-13 16:41:16,584 - INFO - Logging model parameters...
2026-03-13 16:41:16,961 - INFO - Making predictions...
2026-03-13 16:41:16,963 - INFO - Calculating evaluation metrics...
2026-03-13 16:41:17,079 - INFO - Logging evaluation metrics...
2026-03-13 16:41:18,631 - INFO - Saving and logging the model...
2026/03/13 16:41:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 16:41:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run smiling-yak-730 at: https://dagshub.com/Sadat-Shakeeb/production-mlops-sentiment-analysis.mlflow/#/experiments/0/runs/b31968d4e0484021be070e411bd9768e
🧪 View experiment at: https://dagshub.com/Sadat-Shakeeb/production-mlops-sentiment-analysis.mlflow/#/experiments/0
